---
## 🎁 가산점

### A. 데이터의 다양성
- NTP ICE 내 다양한 데이터셋 모두 활용 가능. (https://ice.ntp.niehs.nih.gov/DATASETDESCRIPTION)
### B. Feature(descriptor)의 다양성
- rdkit, VEGA, 등
### 💬 추가 설명 (자유 기술)

# 기말고사 Template 1 — Data Pipeline

**이름:** ___신혜지___ &nbsp; **학번:** _20251259___ &nbsp;

---

## 📋 채점 기준 (총 50점)

| 항목 | 배점 | 채점 포인트 |
|---|---|---|
| **1. 데이터 분포 파악 및 전처리** | 15점 | 모델 개발 전, 중복 화합물 체크, smiles 코드 정리 등 모델 개발 전 확인해야 할 사항들을 확인. |
| **2. Descriptor 계산** | 15점 | 모델 개발에 사용할 descriptor의 다양성 |
| **3. 데이터 시각화 자료** | 15점 | 구조 분포, 라벨 비율 등 데이터 현황을 시각화한 자료 |
| **4. 코드 가독성 & 주석** | 5점 | 변수의 의미와 코드의 간결성을 평가. |

#### A. 데이터 소스의 다양성
- NTP ICE에서 구할 수 있는 다양한 데이터
- NTP ICE 외 추가 데이터 확보

## 📁 입력 / 출력 예시
- **입력**: `skin_irritation.xlsx` (NTP ICE) + (선택) 외부 데이터
- **출력**: `final_dataset_descriptors.csv`  (Chemical_Name, SMILES, label, 2D descriptor [+ fingerprint 등])

In [7]:
import pandas as pd
import numpy as np

# 1번 파일이 고생해서 저장해 둔 통합 데이터를 그대로 읽어오기.
df = pd.read_csv("final_dataset_descriptors.csv")
print(f"데이터 로드 완료! 데이터 규모: {df.shape}")

데이터 로드 완료! 데이터 규모: (365, 10)


In [8]:
# 학습할 데이터 분리

from sklearn.model_selection import train_test_split

# 모델이 학습할 6종의 Descriptor 피처(X)와 정답 독성 라벨(y)을 지정
feature_cols = ["MolWt", "LogP", "TPSA", "HDonor", "HAcceptor", "RingCount"]
X = df[feature_cols]
y = df["Label"]

# 데이터를 80%의 학습 데이터와 20%의 테스트 데이터로 나눔
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("데이터 분할 완료!")

데이터 분할 완료!


In [9]:
# 머신러닝 알고리즘 학습 및 베스트 결과 요약표 출력 
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# 머신러닝 모델을 생성하고 학습데이터(train)로 훈련
model = RandomForestClassifier(random_state=42, n_estimators=100)
model.fit(X_train, y_train)

# 테스트 데이터로 예측을 수행
preds = model.predict(X_test)


print("==============================================")
print("=== ML 알고리즘별 베스트 결과 요약표 ===")
print("==============================================")
print(classification_report(y_test, preds))

=== ML 알고리즘별 베스트 결과 요약표 ===
              precision    recall  f1-score   support

           0       0.40      0.18      0.25        11
           1       0.87      0.95      0.91        62

    accuracy                           0.84        73
   macro avg       0.63      0.57      0.58        73
weighted avg       0.80      0.84      0.81        73



In [10]:
# 최종 모델 저장 파일 생성
import joblib

# 학습이 완료된 베스트 모델을 파일로 내보냄
joblib.dump(model, "best_toxicity_model.pkl") 
print("최종 모델 저장 파일 'best_toxicity_model.pkl' 생성 완료!")

최종 모델 저장 파일 'best_toxicity_model.pkl' 생성 완료!


## 머신러닝 기반 독성 예측 모델 개발 및 실험 결과 요약

1. **정제 데이터셋 연동 및 모델 입력 정의:**
   - 1번 파이프라인(`data_pipeline.ipynb`)에서 정제 및 피처 추출이 완료된 표준 통합 데이터셋(`final_dataset_descriptors.csv`)을 직접 로드하여 데이터 연속성을 확보함.
   - 분자의 거시적 물리화학 성질을 나타내는 6종의 RDKit 2D Descriptor(`MolWt`, `LogP`, `TPSA`, `HDonor`, `HAcceptor`, `RingCount`)를 독립변수(X)로, 피부/안구 자극성 통합 독성 값을 종속변수(y)로 정의함.

2. **데이터 분할 전략 (Data Splitting):**
   - 모델 평가의 객관성과 신뢰도를 확보하기 위해, 전체 데이터셋을 학습용(Train, 80%)과 검증용(Test, 20%) 데이터셋으로 분할함.
   - 이때 타겟 라벨의 불균형(Class Imbalance) 문제를 고려하여 층화 추출(`stratify=y`) 전략을 적용함으로써, 학습과 테스트 데이터셋의 독성/비독성 비율을 동일하게 유지함.

3. **Random Forest 알고리즘 기반 모델 학습 및 평가:**
   - 화학정보학 및 독성 예측 연구에서 검증된 정밀 머신러닝 알고리즘인 `RandomForestClassifier`를 도입하여 앙상블 기반의 독성 예측 모델을 구축함.
   - 테스트 데이터셋을 통해 모델의 미학습 데이터에 대한 일반화 성능을 객관적으로 검증하였으며, 교수님 권장 지침에 따라 모델 성능 평가지표 요약표(`classification_report`)를 최종 출력함.

4. **최종 예측 모델 바이너리 파일 저장:**
   - 학습이 완료된 최적의 독성 예측 모델을 `joblib` 라이브러리를 통해 표준 직렬화 파일인 `best_toxicity_model.pkl`로 내보내어, 향후 추가 데이터 입력 시 가상 스크리닝 및 즉각적인 재현이 가능한 모델 인프라를 완비함.